# 09 — Anagram Patterns (Hashing)

## Learning goals
- Understand anagrams through character frequencies.
- Choose sorting, a frequency dictionary, or a fixed-size array.
- Group anagrams using canonical signatures.
- Connect anagram matching to fixed-size sliding windows.
- Handle character constraints correctly.

**Important:** A 26-element array is valid only when the input is restricted to lowercase English letters (`a`–`z`).

## 1. What is an anagram?

Two strings are anagrams when they contain exactly the same characters with exactly the same frequencies, possibly in a different order.

Examples:
- `listen` and `silent` → anagrams
- `aabb` and `baba` → anagrams
- `aab` and `abb` → not anagrams (frequencies differ)

Order does not matter. Character counts do.

In [1]:
from collections import Counter

print(Counter("listen"))
print(Counter("silent"))
print(Counter("aab"))
print(Counter("abb"))

Counter({'l': 1, 'i': 1, 's': 1, 't': 1, 'e': 1, 'n': 1})
Counter({'s': 1, 'i': 1, 'l': 1, 'e': 1, 'n': 1, 't': 1})
Counter({'a': 2, 'b': 1})
Counter({'b': 2, 'a': 1})


## 2. Clarify the input rules

Before coding, check:
- Is matching case-sensitive?
- Should spaces count?
- Should punctuation count?
- Is input limited to lowercase English letters?
- Can it contain Unicode characters?

Do not silently normalize input unless the problem asks for it.

In [2]:
def normalize_for_anagram(s: str) -> str:
    # Example policy: ignore non-alphanumeric characters and case.
    return "".join(ch.lower() for ch in s if ch.isalnum())

a = normalize_for_anagram("Dormitory!")
b = normalize_for_anagram("Dirty room")
print(a, b)
print(Counter(a) == Counter(b))

dormitory dirtyroom
True


## 3. Approach 1 — Sort both strings

If two strings are anagrams, sorting their characters produces the same result.

Example: `listen` → `eilnst`, `silent` → `eilnst`.

```python
def are_anagrams_sorting(s: str, t: str) -> bool:
    return sorted(s) == sorted(t)
```

Complexity for lengths `n` and `m`:
- Time: `O(n log n + m log m)`
- Extra space: typically `O(n + m)` for Python's sorted lists.

This is simple and often a good first solution.

In [3]:
def are_anagrams_sorting(s: str, t: str) -> bool:
    return sorted(s) == sorted(t)

print(are_anagrams_sorting("listen", "silent"))  # True
print(are_anagrams_sorting("rat", "car"))        # False
print(are_anagrams_sorting("aab", "abb"))        # False

True
False
False


## 4. Approach 2 — Frequency dictionary

Anagrams must have the same frequency for every character.

Steps:
1. If lengths differ, return `False`.
2. Count each character in the first string.
3. Subtract counts using the second string.
4. If a character is missing or a count becomes negative, return `False`.
5. Otherwise, return `True`.

This works for general hashable characters; it does not assume only `a`–`z`.

In [4]:
def are_anagrams_frequency(s: str, t: str) -> bool:
    if len(s) != len(t):
        return False

    freq = {}
    for ch in s:
        freq[ch] = freq.get(ch, 0) + 1

    for ch in t:
        if ch not in freq:
            return False
        freq[ch] -= 1
        if freq[ch] < 0:
            return False

    return True

print(are_anagrams_frequency("listen", "silent"))  # True
print(are_anagrams_frequency("aab", "abb"))        # False
print(are_anagrams_frequency("a", "aa"))           # False

True
False
False


## 5. Approach 3 — `Counter`

`collections.Counter` is a frequency map designed for counting hashable items. It is concise and readable.

In [5]:
from collections import Counter

def are_anagrams_counter(s: str, t: str) -> bool:
    return Counter(s) == Counter(t)

print(are_anagrams_counter("anagram", "nagaram"))  # True
print(are_anagrams_counter("hello", "bello"))      # False

True
False


## 6. Approach 4 — Fixed-size frequency array

Use this only when the problem guarantees lowercase English letters.

There are 26 possible characters:
- `a` maps to index `0`
- `b` maps to index `1`
- ...
- `z` maps to index `25`

Index formula: `ord(ch) - ord('a')`.

Count the first string by adding 1 and the second by subtracting 1. All 26 counts must finish at zero.

In [6]:
def are_anagrams_lowercase(s: str, t: str) -> bool:
    if len(s) != len(t):
        return False

    freq = [0] * 26

    for ch in s:
        freq[ord(ch) - ord('a')] += 1

    for ch in t:
        freq[ord(ch) - ord('a')] -= 1

    return all(count == 0 for count in freq)

print(are_anagrams_lowercase("anagram", "nagaram"))  # True
print(are_anagrams_lowercase("rat", "car"))          # False

True
False


## 7. Dry run — frequency array

Input: `s = "aab"`, `t = "aba"`.

Start with 26 zeroes.
- Read `s`: `a` adds 1, another `a` adds 1, `b` adds 1.
- Read `t`: `a` subtracts 1, `b` subtracts 1, `a` subtracts 1.
- Every count returns to zero: anagrams.

For `s = "aab"`, `t = "abb"`, final counts are `a: +1` and `b: -1`: not anagrams.

In [7]:
def nonzero_frequency_counts(s: str, t: str) -> dict:
    freq = [0] * 26

    for ch in s:
        freq[ord(ch) - ord('a')] += 1
    for ch in t:
        freq[ord(ch) - ord('a')] -= 1

    return {chr(i + 97): count for i, count in enumerate(freq) if count}

print(nonzero_frequency_counts("aab", "aba"))  # {}
print(nonzero_frequency_counts("aab", "abb"))  # {'a': 1, 'b': -1}

{}
{'a': 1, 'b': -1}


## 8. Group Anagrams — canonical signatures

Given a list of words, place anagrams in the same group.

A **signature** is a representation that is identical for all anagrams.

### Signature A: sorted characters
`eat`, `tea`, and `ate` all become `aet`.

### Signature B: frequency tuple
For lowercase English letters, count all 26 letters and use the tuple as a dictionary key. This avoids sorting each word.

In [8]:
from collections import defaultdict

def group_anagrams_sorted(words):
    groups = defaultdict(list)

    for word in words:
        key = tuple(sorted(word))
        groups[key].append(word)

    return list(groups.values())

words = ["eat", "tea", "tan", "ate", "nat", "bat"]
print(group_anagrams_sorted(words))

[['eat', 'tea', 'ate'], ['tan', 'nat'], ['bat']]


### Grouping with a 26-count signature

This version assumes every word contains only lowercase English letters.

In [9]:
from collections import defaultdict

def group_anagrams_frequency(words):
    groups = defaultdict(list)

    for word in words:
        count = [0] * 26
        for ch in word:
            count[ord(ch) - ord('a')] += 1
        groups[tuple(count)].append(word)

    return list(groups.values())

words = ["eat", "tea", "tan", "ate", "nat", "bat"]
print(group_anagrams_frequency(words))

[['eat', 'tea', 'ate'], ['tan', 'nat'], ['bat']]


## 9. Find all anagram starting indices — sliding window

Given strings `s` and `p`, return every starting index in `s` where a substring is an anagram of `p`.

A matching substring must have:
- Exactly `len(p)` characters.
- Exactly the same character frequencies as `p`.

For lowercase English letters:
1. Count the pattern.
2. Build a window of size `len(p)`.
3. Compare its 26 counts with the pattern.
4. Slide: remove the outgoing character and add the incoming character.

Since 26 is fixed, comparing two 26-element arrays is constant time.

In [10]:
def find_anagrams(s: str, p: str) -> list[int]:
    if len(p) > len(s):
        return []

    need = [0] * 26
    window = [0] * 26

    for ch in p:
        need[ord(ch) - 97] += 1

    k = len(p)
    result = []

    for i, ch in enumerate(s):
        window[ord(ch) - 97] += 1

        if i >= k:
            outgoing = s[i - k]
            window[ord(outgoing) - 97] -= 1

        if i >= k - 1 and window == need:
            result.append(i - k + 1)

    return result

print(find_anagrams("cbaebabacd", "abc"))  # [0, 6]
print(find_anagrams("abab", "ab"))         # [0, 1, 2]

[0, 6]
[0, 1, 2]


## 10. Permutation in String

A string `s1` is a permutation of some substring of `s2` exactly when a window of length `len(s1)` has the same character frequencies as `s1`.

This is the same core pattern as finding all anagram indices. We return a boolean and stop at the first match.

In [11]:
def check_inclusion(s1: str, s2: str) -> bool:
    if len(s1) > len(s2):
        return False

    need = [0] * 26
    window = [0] * 26

    for ch in s1:
        need[ord(ch) - 97] += 1

    k = len(s1)

    for i, ch in enumerate(s2):
        window[ord(ch) - 97] += 1

        if i >= k:
            window[ord(s2[i - k]) - 97] -= 1

        if i >= k - 1 and window == need:
            return True

    return False

print(check_inclusion("ab", "eidbaooo"))  # True
print(check_inclusion("ab", "eidboaoo"))  # False

True
False


## 11. Advanced anagram-related patterns

### Minimum steps to make two strings anagrams
When replacing characters in one equal-length string, compare frequencies. The surplus in the target string tells how many replacements are required.

### Determine if two strings are close
This uses two conditions:
1. Both strings have the same set of characters.
2. Their character-frequency multisets are equal (counts can be reassigned between characters).

This is not ordinary anagram checking.

### Anagram mapping
If values repeat, store a list of available indices for each value rather than only one index. Otherwise duplicates can be lost.

In [12]:
def min_steps_to_anagram(s: str, t: str) -> int:
    # Assumes equal lengths; replace characters in t.
    return sum((Counter(s) - Counter(t)).values())

print(min_steps_to_anagram("bab", "aba"))  # 1

def close_strings(word1: str, word2: str) -> bool:
    if len(word1) != len(word2):
        return False

    c1 = Counter(word1)
    c2 = Counter(word2)

    return set(c1) == set(c2) and sorted(c1.values()) == sorted(c2.values())

print(close_strings("abc", "bca"))        # True
print(close_strings("a", "aa"))           # False
print(close_strings("cabbba", "abbccc"))  # True

1
True
False
True


## 12. Choosing the right approach

| Situation | Good approach |
|---|---|
| Simple anagram check | Sorting |
| General character set | Dictionary / `Counter` |
| Lowercase `a`–`z` guaranteed | 26-element frequency array |
| Group many anagrams | Dictionary keyed by signature |
| Find anagrams inside another string | Fixed-size sliding window + frequencies |
| Compare only frequency distributions | `Counter` plus set/sorted counts |

### Complexity
Let `n`, `m` be string lengths and `k` be the fixed alphabet size (26):
- Sorting: `O(n log n + m log m)` time.
- Frequency dictionary: expected `O(n + m)` time; space proportional to distinct characters.
- Fixed array: `O(n + m + k)` time and `O(k)` extra space.
- Grouping by sorted signature: approximately `O(W * L log L)` for `W` words of average length `L`.
- Grouping by 26-count signature: `O(W * (L + 26))`.
- Fixed-window search with full 26-count comparisons: `O(N * 26)`, which is `O(N)` for a fixed alphabet; `O(26)` extra space.

Python dictionary operations are expected `O(1)` on average, not guaranteed worst-case `O(1)`.

## 13. Common mistakes

1. Checking only distinct characters. `aab` and `abb` show why this fails.
2. Forgetting that frequencies matter.
3. Using 26 slots when uppercase, punctuation, or arbitrary Unicode is allowed.
4. Ignoring case or spaces without being asked.
5. Forgetting that the window size must equal the pattern length.
6. Adding the incoming character but forgetting to remove the outgoing character.
7. Storing only one index when duplicate values require multiple indices.
8. Confusing exact per-character frequency equality with equality of frequency distributions (as in Close Strings).

## 14. Interview recognition checklist

When you see:
- “same characters in a different order”
- “rearrangement” or “permutation”
- “group words that contain the same letters”
- “find every substring that is a permutation”
- “minimum replacements to match character counts”

Ask:
1. Does order matter? If not, frequency counting may help.
2. Are the character constraints fixed and small?
3. Is this one comparison, grouping many strings, or scanning windows?
4. If scanning windows, is the window size fixed?
5. Do I need exact per-character counts, or only the distribution of counts?

**Core idea:** Anagrams are about character frequencies. The problem tells you whether to compare, group, or maintain those frequencies in a moving window.

## 15. Practice questions — solve before looking at solutions

### Easy
1. [242. Valid Anagram](https://leetcode.com/problems/valid-anagram/)
2. [2273. Find Resultant Array After Removing Anagrams](https://leetcode.com/problems/find-resultant-array-after-removing-anagrams/)

### Medium
3. [49. Group Anagrams](https://leetcode.com/problems/group-anagrams/)
4. [438. Find All Anagrams in a String](https://leetcode.com/problems/find-all-anagrams-in-a-string/)
5. [567. Permutation in String](https://leetcode.com/problems/permutation-in-string/)
6. [1347. Minimum Number of Steps to Make Two Strings Anagram](https://leetcode.com/problems/minimum-number-of-steps-to-make-two-strings-anagram/)
7. [1657. Determine if Two Strings Are Close](https://leetcode.com/problems/determine-if-two-strings-are-close/)

Suggested order: 242 → 2273 → 49 → 1347 → 1657 → 567 → 438.

For each problem: state brute force, identify the required frequency information, choose a data structure, explain correctness, test edge cases, and state complexity.

## 16. Reusable templates

### General frequency comparison
```python
from collections import Counter

def same_frequencies(a: str, b: str) -> bool:
    return Counter(a) == Counter(b)
```

### Lowercase English signature
```python
def lowercase_signature(s: str) -> tuple[int, ...]:
    count = [0] * 26
    for ch in s:
        count[ord(ch) - ord('a')] += 1
    return tuple(count)
```

### Group by signature
```python
from collections import defaultdict

def group_by_signature(words: list[str]) -> list[list[str]]:
    groups = defaultdict(list)
    for word in words:
        groups[lowercase_signature(word)].append(word)
    return list(groups.values())
```

### Fixed-size sliding window
```python
# 1. Add the new right character.
# 2. If window length exceeds k, remove the left character.
# 3. Compare window counts with target counts.
```

You should now be able to recognize the anagram family and select the right frequency-based pattern.